# Pointer Networks (Ptr-Net) — Implementation

This notebook covers the code implementation, training, and benchmarks.
For theory and mathematical foundations see **`notebook_theory.ipynb`**.

**Files:**
```
model.py   — Encoder, Attention, PointerNetwork
data.py    — data helpers and dataset loader
train.py   — training loop (run from terminal)
```

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║                     MODEL CONFIGURATION                             ║
# ║  Choose the size to use throughout this notebook.                   ║
# ║  Run the matching train.py command first if you haven't already.    ║
# ║                                                                      ║
# ║  Commands:                                                           ║
# ║    python train.py --size small  --n 8 --label optimal --steps 1000  --out model/ptr_net_small.pt  ║
# ║    python train.py --size medium --n 8 --label optimal --steps 3000  --out model/ptr_net_medium.pt ║
# ║    python train.py --size large  --n 8 --label optimal --steps 5000  --out model/ptr_net_large.pt  ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── Select model size ────────────────────────────────────────────────────────
#   'small'  — embed=64,  hidden=128, layers=1  (~232K params)   quick tests, n ≤ 20
#   'medium' — embed=128, hidden=256, layers=1  (~923K params)   standard TSP, n ≤ 50  ← paper default
#   'large'  — embed=256, hidden=512, layers=2  (~3.7M params)   larger instances
SIZE       = "medium"
MODEL_PATH = f"model/ptr_net_{SIZE}.pt"

print(f"Using model: {SIZE}  →  {MODEL_PATH}")

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
from itertools import permutations
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from data import random_instance, tour_length, optimal_tour, load_cities, nn_tour
from model import PointerNetwork, MODEL_SIZES

print('Modules imported successfully.')
print('\nParameter counts per preset:')
for size, (e, h, l) in MODEL_SIZES.items():
    n_params = sum(p.numel() for p in PointerNetwork(e, h, l).parameters())
    print(f'  {size:<8} embed={e:>3}, hidden={h:>3}, layers={l}  →  {n_params:>10,} parameters')

# ── Figures output directory ─────────────────────────────────────────────────
# All plots are saved here for inclusion in the LaTeX documentation.
# (docs/src/doc_ptr_net.tex references these files via \graphicspath)
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)
print(f'Figures will be saved to: {FIGURES_DIR.resolve()}')

## 7. Implementation

| File | Role |
|------|------|
| `model.py` | `Encoder`, `Attention`, `PointerNetwork` — LSTM encoder-decoder with pointer head |
| `data.py` | `random_instance`, `optimal_tour`, `nn_tour`, `tour_length`, `load_cities` |
| `train.py` | Supervised training loop — NLL loss with teacher forcing (CLI + importable `train()`) |

## 8. Demonstration

> The model is trained externally via `train.py`. Run one of the commands in the config cell above,
> then execute the cells below to load the saved model and visualise the result.

```bash
# Quick test (small model, n=8 brute-force labels)
python train.py --size small --n 8 --label optimal --steps 1000

# Standard training (medium model, optimal labels)
python train.py --size medium --n 8 --label optimal --steps 3000 --source tsp

# OOD generalisation (fine-tune with nn2opt labels at larger n)
python train.py --resume model/ptr_net_medium.pt --size medium \
                --n 50 --label nn2opt --source tsp --steps 2000 --lr 5e-4
```

In [ ]:
# ── Load trained model (SIZE and MODEL_PATH set in config cell) ──────────────
embed, hidden, n_layers = MODEL_SIZES[SIZE]
model_demo = PointerNetwork(embed_dim=embed, hidden_dim=hidden, n_layers=n_layers)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"{MODEL_PATH} not found — run train.py first.")

model_demo.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model_demo.eval()
losses = np.load('model/losses.npy').tolist()
print(f'Loaded {SIZE} model ({sum(p.numel() for p in model_demo.parameters()):,} params)'
      f' — {len(losses)} training steps, best loss {min(losses):.4f}')

# ── Inference on one held-out instance ───────────────────────────────────────
N_DEMO = 8
torch.manual_seed(7)
coords   = random_instance(N_DEMO)
opt_tour = optimal_tour(coords)
opt_len  = tour_length(coords, opt_tour)

with torch.no_grad():
    log_probs, ptr_tour = model_demo(coords)

ptr_len = tour_length(coords, ptr_tour)
gap     = (ptr_len - opt_len) / opt_len * 100
print(f'Optimal tour length  : {opt_len:.4f}')
print(f'Ptr-Net greedy tour  : {ptr_len:.4f}  (gap = {gap:.1f} %)')

# ── Visualisation ─────────────────────────────────────────────────────────────
# 4 panels matching the GNN demo:
#   (1) Training loss curve
#   (2) Pointer probability matrix  — log_probs[t, i] = log P(π_t = i | ...)
#   (3) Optimal tour (green)
#   (4) Ptr-Net tour (blue) with gap
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].plot(losses, color='steelblue', lw=1.2)
axes[0].set_title('Training Loss (NLL)')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

# Pointer probability matrix: exp(log_probs) → shape (n, n)
# Row t = probability distribution over cities at decoding step t
prob_matrix = torch.exp(log_probs).detach().numpy()
im = axes[1].imshow(prob_matrix, cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_title('Pointer probs $P(\\pi_t = i \\mid \\cdots)$')
axes[1].set_xlabel('City $i$')
axes[1].set_ylabel('Decoding step $t$')
plt.colorbar(im, ax=axes[1], fraction=0.046)

xy = coords.numpy()

def draw_tour(ax, xy, tour, color, lw, label):
    tc = tour + [tour[0]]
    for k in range(len(tc) - 1):
        a, b = tc[k], tc[k + 1]
        ax.plot([xy[a, 0], xy[b, 0]], [xy[a, 1], xy[b, 1]],
                color=color, lw=lw, alpha=0.8, label=label if k == 0 else None)

for ax, tour, color, title in [
    (axes[2], opt_tour,  'green',     f'Optimal ({opt_len:.3f})'),
    (axes[3], ptr_tour,  'steelblue', f'Ptr-Net ({ptr_len:.3f})  gap={gap:.1f}%'),
]:
    draw_tour(ax, xy, tour, color, 2.0, title)
    ax.scatter(xy[:, 0], xy[:, 1], s=80, zorder=5, color='black')
    ax.scatter(xy[0, 0], xy[0, 1], s=120, zorder=6, color='gold',
               edgecolors='black', linewidths=0.8, label='Start')
    for i, (xi, yi) in enumerate(xy):
        ax.annotate(str(i), (xi + 0.02, yi + 0.02), fontsize=9)
    ax.set_title(title)
    ax.set_xlim(-0.05, 1.10)
    ax.set_ylim(-0.05, 1.10)
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'demo_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {FIGURES_DIR / "demo_overview.png"}')
print(f'\nPtr-Net tour : {" → ".join(map(str, ptr_tour + [ptr_tour[0]]))}')
print(f'Optimal      : {" → ".join(map(str, opt_tour  + [opt_tour[0]]))}')

## 9. Benchmark

> Benchmark sizes: **n = 1 / 10 / 50 / 100 / 200** — fully executed.
> Sizes n ≥ 1 000 are feasible memory-wise (encoder memory is O(n·d) — linear),
> but the sequential LSTM decoder becomes a throughput bottleneck.
> Generalisation for n > 10 is **out-of-distribution** for a model trained on n ≤ 10.

### Metrics

| Metric | Definition |
|--------|------------|
| **% success** | Runs where gap to reference ≤ 1 % |
| **% non-detection** | Model always outputs a tour but gap > 1 % — a *silent failure*; equals 100 − % success |
| **% false detection** | Structurally 0 % — the model never refuses to output a tour |
| **Inference time** | Mean time for one forward pass + greedy decode (CPU, 20 repetitions) |

### Reference tour per size

| n | Reference |
|---|-----------|
| 1 | Trivial (single city, zero-length tour) |
| 10 | Brute-force exact optimal |
| 50, 100, 200 | Nearest-neighbour heuristic |

All figures are saved to `figures/` for direct inclusion in `docs/src/doc_ptr_net.tex`.

In [ ]:
# ── Benchmark: n = 1 / 10 / 50 / 100 / 200 ──────────────────────────────────
# Random weights are used for timing (isolates compute cost from solution quality).
# Trained weights are used for quality metrics (% success / % non-detection).
#
# Memory column: encoder memory = n * hidden_dim * 4 bytes (O(n·d), linear in n)
# This contrasts with the GNN's O(n²·d) edge tensor.

N_REPS_TIME = 20   # repetitions for timing
N_REPS_QUAL = 50   # instances for quality statistics

embed, hidden, n_layers = MODEL_SIZES[SIZE]

model_bench = PointerNetwork(embed_dim=embed, hidden_dim=hidden,
                              n_layers=n_layers).eval()   # random weights — timing
model_qual  = PointerNetwork(embed_dim=embed, hidden_dim=hidden,
                              n_layers=n_layers).eval()   # trained weights — quality
if os.path.exists(MODEL_PATH):
    model_qual.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
else:
    print(f'WARNING: {MODEL_PATH} not found — quality metrics will use random weights')

def _nn_tour(coords):
    return nn_tour(coords)

BENCH_SIZES = [1, 10, 50, 100, 200]
results = {}

hdr = (f"{'n':>6}  {'time_ms':>10}  {'±ms':>7}  {'enc_MB':>9}"
       f"  {'success%':>10}  {'non_det%':>10}  {'false_det%':>12}")
print(hdr)
print('-' * len(hdr))

with torch.no_grad():
    for n in BENCH_SIZES:
        # ── Timing ───────────────────────────────────────────────────────────
        times = []
        for _ in range(N_REPS_TIME):
            x  = random_instance(n) if n > 1 else torch.rand(1, 2)
            t0 = time.perf_counter()
            _, _ = model_bench(x)
            times.append(time.perf_counter() - t0)

        mean_ms = float(np.mean(times)) * 1e3
        std_ms  = float(np.std(times))  * 1e3
        # Encoder memory: n hidden states × hidden_dim × 4 bytes (float32)
        enc_mb  = n * hidden * 4 / 1e6

        # ── Quality metrics ───────────────────────────────────────────────────
        if n == 1:
            success_pct   = 100.0
            non_det_pct   = 0.0
            false_det_pct = 0.0
        else:
            gaps = []
            for _ in range(N_REPS_QUAL):
                x = random_instance(n)
                _, ptr_tour = model_qual(x)
                ptr_len = float(tour_length(x, ptr_tour))

                if n <= 10:
                    best = float('inf')
                    for perm in permutations(range(1, n)):
                        t = [0] + list(perm)
                        l = float(tour_length(x, t))
                        if l < best:
                            best = l
                    ref_len = best
                else:
                    ref_len = float(tour_length(x, _nn_tour(x)))

                gaps.append((ptr_len - ref_len) / max(ref_len, 1e-9) * 100.0)

            # Model is deterministic and never signals failure → false-detection = 0 %.
            success_pct   = float(np.mean([g <= 1.0 for g in gaps])) * 100
            non_det_pct   = 100.0 - success_pct
            false_det_pct = 0.0

        results[n] = {
            'mean_ms': mean_ms, 'std_ms': std_ms, 'enc_mb': enc_mb,
            'success_pct': success_pct, 'non_det_pct': non_det_pct,
            'false_det_pct': false_det_pct,
        }
        print(f"{n:>6}  {mean_ms:>10.2f}  {std_ms:>7.2f}  {enc_mb:>9.4f}"
              f"  {success_pct:>10.1f}  {non_det_pct:>10.1f}  {false_det_pct:>12.1f}")

# ── Plot 1: computational benchmark ──────────────────────────────────────────
ns    = BENCH_SIZES
means = [results[n]['mean_ms'] for n in ns]
stds  = [results[n]['std_ms']  for n in ns]
mems  = [results[n]['enc_mb']  for n in ns]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'PointerNetwork {SIZE} (embed={embed}, hidden={hidden}) — Computational Benchmark',
             fontsize=12)

# Inference time
axes[0].errorbar(ns, means, yerr=stds, marker='o', capsize=4, color='steelblue', lw=1.5)
axes[0].set_xlabel('Number of cities $n$')
axes[0].set_ylabel('Inference time (ms)')
axes[0].set_title('Inference Time')
axes[0].grid(True, alpha=0.3)

# Log-log: time ~ O(n²) due to attention; memory ~ O(n) linear
ns_arr = np.array(ns, dtype=float)
ref_n2 = means[0] * (ns_arr / ns_arr[0]) ** 2
axes[1].loglog(ns_arr, means,  'o-',  color='steelblue', lw=1.5, label='Time (measured)')
axes[1].loglog(ns_arr, ref_n2, 'r--', lw=1.2,            label=r'$O(n^2)$ ref.')
axes[1].set_xlabel('$n$  (log scale)')
axes[1].set_ylabel('Time (ms, log scale)')
axes[1].set_title('Log-log Scaling')
axes[1].legend(fontsize=9)
axes[1].grid(True, which='both', alpha=0.3)

# Encoder memory (linear in n — key advantage over GNN)
axes[2].plot(ns, mems, 'o-', color='darkorange', lw=1.5)
axes[2].set_xlabel('Number of cities $n$')
axes[2].set_ylabel('Encoder memory (MB)')
axes[2].set_title(f'Memory Footprint — O(n·d)  (hidden={hidden})')
axes[2].grid(True, alpha=0.3)
for n, m in zip(ns, mems):
    axes[2].annotate(f'{m:.3f} MB', (n, m), textcoords='offset points',
                     xytext=(6, 4), fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'benchmark_computation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {FIGURES_DIR / "benchmark_computation.png"}')

# ── Plot 2: quality metrics bar chart ────────────────────────────────────────
fig2, ax = plt.subplots(figsize=(7, 4))
x_pos   = np.arange(len(ns))
w       = 0.35
s_vals  = [results[n]['success_pct'] for n in ns]
nd_vals = [results[n]['non_det_pct'] for n in ns]

b1 = ax.bar(x_pos - w/2, s_vals,  w, label='% success (gap ≤ 1 %)', color='steelblue')
b2 = ax.bar(x_pos + w/2, nd_vals, w, label='% non-detection (gap > 1 %)', color='tomato')
ax.set_xticks(x_pos)
ax.set_xticklabels([str(n) for n in ns])
ax.set_xlabel('Number of cities $n$')
ax.set_ylabel('Proportion (%)')
ax.set_title(f'Quality Metrics — Ptr-Net {SIZE}   (false-detection = 0 % by construction)\n'
             f'Note: n > 10 is out-of-distribution (model trained on n ≤ 10)')
ax.set_ylim(0, 120)
ax.legend(fontsize=9)
ax.bar_label(b1,  fmt='%.0f%%', fontsize=8, padding=2)
ax.bar_label(b2,  fmt='%.0f%%', fontsize=8, padding=2)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'benchmark_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {FIGURES_DIR / "benchmark_quality.png"}')

# ── Scalability projection ────────────────────────────────────────────────────
print('\n--- Scalability projection beyond benchmark range ---')
t200 = results[200]['mean_ms']
m200 = results[200]['enc_mb']
print(f"{'n':>10}  {'proj. time':>14}  {'enc memory':>14}  {'memory ok?':>12}")
print('-' * 56)
for n_proj in [500, 1_000, 5_000, 10_000, 100_000]:
    # Time scales as O(n²) due to attention loop
    scale_t = (n_proj / 200) ** 2
    # Memory scales as O(n) — linear
    scale_m = n_proj / 200
    t_proj  = t200 * scale_t
    m_proj  = m200 * scale_m
    unit_t  = 'ms' if t_proj < 1000 else ('s' if t_proj < 60000 else 'min')
    val_t   = t_proj if unit_t == 'ms' else (t_proj/1e3 if unit_t == 's' else t_proj/60e3)
    mem_ok  = 'Yes' if m_proj < 1024 else ('16 GB+' if m_proj < 16384 else 'Extreme')
    print(f"{n_proj:>10,}  {val_t:>10.1f} {unit_t:>3}  {m_proj:>10.1f} MB  {mem_ok:>12}")
print('\nNote: memory stays feasible (O(n·d)) but time grows as O(n²).')
print('      Quality degrades for n >> training size (generalisation gap).')

## 9b. Solution Quality on Real Dataset

> Tests the **trained model** on instances loaded from the CSV dataset.
> Each instance is compared against a **nearest-neighbour (NN) baseline**.

### Metrics

| Metric | Formula | Meaning |
|--------|---------|--------|
| **Tour length** | $\sum_k d(c_k, c_{k+1})$ | Total Euclidean distance of the decoded tour |
| **Gap vs NN** | $(L_{\text{Ptr}} - L_{\text{NN}}) / L_{\text{NN}} \times 100\%$ | Relative difference; negative = Ptr-Net beats NN |
| **Win rate** | $\#\{L_{\text{Ptr}} \le L_{\text{NN}}\} / N$ | Fraction of instances where Ptr-Net is better or equal |

> **Note:** run `python train.py` from `DL_MODEL/ptr_net/` before executing this cell.

In [ ]:
# ── Load trained model on best available device ──────────────────────────────
from train import get_device
bench_device = get_device()
print(f'Ptr-Net will run on: {bench_device}')
print(f'NN      will run on: cpu (sequential by nature)')

model_test = PointerNetwork(embed_dim=embed, hidden_dim=hidden,
                             n_layers=n_layers).to(bench_device)
if os.path.exists(MODEL_PATH):
    model_test.load_state_dict(torch.load(MODEL_PATH, map_location=bench_device))
    print(f'Loaded weights from {MODEL_PATH}')
else:
    print(f'WARNING: {MODEL_PATH} not found — using random weights')
model_test.eval()

# ── Load cities from dataset ──────────────────────────────────────────────────
EVAL_SIZES = [10, 50, 100, 200]
N_REPS_GEN = 10
instances  = []
for n in EVAL_SIZES:
    coords = load_cities(n, source='tsp')
    instances.append({'n': n, 'coords': coords})
    print(f'  n={n:>4}  shape={tuple(coords.shape)}')

# ── Nearest-neighbour baseline (always CPU) ───────────────────────────────────
def nearest_neighbour_tour(coords):
    return nn_tour(coords)

# ── Inference + generation time ───────────────────────────────────────────────
records = []
with torch.no_grad():
    for inst in instances:
        coords     = inst['coords']
        coords_dev = coords.to(bench_device)

        # Ptr-Net: warm up, then time N_REPS_GEN runs
        model_test(coords_dev)
        if bench_device.type == 'cuda':
            torch.cuda.synchronize()
        ptr_times = []
        for _ in range(N_REPS_GEN):
            t0 = time.perf_counter()
            _, ptr_tour = model_test(coords_dev)
            if bench_device.type == 'cuda':
                torch.cuda.synchronize()
            ptr_times.append(time.perf_counter() - t0)
        ptr_ms = float(np.mean(ptr_times)) * 1e3

        # NN: always CPU
        nn_times = []
        for _ in range(N_REPS_GEN):
            t0 = time.perf_counter()
            nn_t = nearest_neighbour_tour(coords)
            nn_times.append(time.perf_counter() - t0)
        nn_ms = float(np.mean(nn_times)) * 1e3

        ptr_len = tour_length(coords, ptr_tour)
        nn_len  = tour_length(coords, nn_t)
        gap     = (ptr_len - nn_len) / nn_len * 100

        records.append({
            'n': inst['n'], 'coords': coords,
            'ptr_tour': ptr_tour, 'nn_tour': nn_t,
            'ptr': ptr_len, 'nn': nn_len, 'gap': gap,
            'ptr_ms': ptr_ms, 'nn_ms': nn_ms,
        })

# ── Results table ─────────────────────────────────────────────────────────────
dev_label = str(bench_device)
print(f'\nPtr-Net device: {dev_label}  |  NN device: cpu')
print(f"{'n':>6}  {'Ptr tour':>10}  {'NN tour':>10}  {'gap %':>8}"
      f"  {'Ptr ms':>8}  {'NN ms':>8}  {'speedup':>8}  {'better':>6}")
print('-' * 72)
for r in records:
    better  = 'Ptr' if r['ptr'] <= r['nn'] else 'NN '
    speedup = r['nn_ms'] / max(r['ptr_ms'], 1e-9)
    print(f"{r['n']:>6}  {r['ptr']:>10.4f}  {r['nn']:>10.4f}  {r['gap']:>+8.2f}%"
          f"  {r['ptr_ms']:>8.2f}  {r['nn_ms']:>8.2f}  {speedup:>7.1f}x  {better:>6}")

# ── Tour visualisation ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, len(records), figsize=(5 * len(records), 9))

def plot_tour(ax, xy, tour, color, lw, label):
    closed = tour + [tour[0]]
    for k in range(len(closed) - 1):
        a, b = closed[k], closed[k + 1]
        ax.plot([xy[a, 0], xy[b, 0]], [xy[a, 1], xy[b, 1]],
                color=color, lw=lw, alpha=0.8,
                label=label if k == 0 else None)

for col, r in enumerate(records):
    xy = r['coords'].numpy()
    for row, (tour, color, title) in enumerate([
        (r['ptr_tour'], 'steelblue', f"Ptr-Net [{dev_label}]  len={r['ptr']:.3f}  {r['ptr_ms']:.1f} ms"),
        (r['nn_tour'],  'tomato',    f"NN  [cpu]  len={r['nn']:.3f}  {r['nn_ms']:.1f} ms"),
    ]):
        ax = axes[row][col]
        plot_tour(ax, xy, tour, color=color, lw=1.5, label=title)
        ax.scatter(xy[:, 0], xy[:, 1], s=20, zorder=5, color='black')
        ax.scatter(xy[0, 0], xy[0, 1], s=80, zorder=6, color='gold',
                   edgecolors='black', linewidths=0.8)
        ax.set_title(f"n={r['n']} — {title}  (gap {r['gap']:+.1f}%)", fontsize=9)
        ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
        ax.set_aspect('equal'); ax.grid(True, alpha=0.2)

fig.suptitle(f'Ptr-Net [{dev_label}] vs NN [cpu] — TSP dataset cities', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'solution_quality_tours.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {FIGURES_DIR / "solution_quality_tours.png"}')

# ── Generation time comparison ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Generation Time — Ptr-Net [{dev_label}] vs NN [cpu]', fontsize=12)

ns_r     = [r['n']      for r in records]
ptr_mss  = [r['ptr_ms'] for r in records]
nn_mss   = [r['nn_ms']  for r in records]
speedups = [r['nn_ms'] / max(r['ptr_ms'], 1e-9) for r in records]

axes[0].plot(ns_r, ptr_mss, 'o-',  color='steelblue', lw=1.5, label=f'Ptr-Net [{dev_label}]')
axes[0].plot(ns_r, nn_mss,  's--', color='tomato',    lw=1.5, label='NN [cpu]')
axes[0].set_xlabel('Number of cities $n$')
axes[0].set_ylabel('Generation time (ms)')
axes[0].set_title('Generation Time vs n')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

colors = ['steelblue' if s >= 1.0 else 'tomato' for s in speedups]
bars   = axes[1].bar([str(n) for n in ns_r], speedups, color=colors, edgecolor='white')
axes[1].axhline(1.0, color='black', lw=0.8, ls='--')
axes[1].bar_label(bars, fmt='%.1fx', fontsize=9, padding=3)
axes[1].set_xlabel('Number of cities $n$')
axes[1].set_ylabel(f'Speedup  (NN cpu / Ptr-Net {dev_label}, >1 = Ptr-Net faster)')
axes[1].set_title('Ptr-Net speedup over NN')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'generation_time.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {FIGURES_DIR / "generation_time.png"}')

# ── Gap summary bar chart ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
gaps_r  = [r['gap'] for r in records]
colors  = ['steelblue' if g <= 0 else 'tomato' for g in gaps_r]
bars    = ax.bar([str(r['n']) for r in records], gaps_r, color=colors, edgecolor='white')
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.bar_label(bars, fmt='%.1f%%', fontsize=9, padding=3)
ax.set_xlabel('Number of cities $n$')
ax.set_ylabel('Gap vs NN (%)')
ax.set_title(f'Ptr-Net ({SIZE}) — gap vs. NN heuristic  (blue = Ptr-Net wins)')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'solution_quality_gap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {FIGURES_DIR / "solution_quality_gap.png"}')